# Program 3 – Softmax and Cross-Entropy Demonstration

**Objective:** Implement softmax and categorical cross-entropy manually (NumPy) and verify against TensorFlow / Keras.

**Dataset:** MNIST (10 classes)

**Key tasks**
1. Use NumPy to implement softmax + categorical cross-entropy.
2. Verify results with TensorFlow's `tf.nn.softmax_cross_entropy_with_logits`.

## Theory (for viva)

**Logits** $z = (z_1, \dots, z_K)$: raw, unnormalized scores from the last layer.

**Softmax** turns logits into probabilities that sum to 1:
$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

**Numerical stability trick:** $e^{z}$ overflows for large $z$ (e.g. $e^{1000} = \infty$). Since softmax is unchanged when a constant is subtracted from every logit, we compute
$$\text{softmax}(z)_i = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$

**Categorical cross-entropy** between one-hot true label $y$ and predicted probabilities $p$:
$$L = -\sum_{i=1}^{K} y_i \log p_i = -\log p_{\text{true class}}$$
Batch loss = mean over samples.

**Why "with logits"?** Computing softmax then log separately can give $\log(0) = -\infty$. TensorFlow fuses them using log-sum-exp:
$$L = -z_{\text{true}} + \log \sum_j e^{z_j}$$
which is stable.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

np.set_printoptions(precision=6, suppress=True)  # Print floats with 6 decimals, no scientific notation

## Step 2 – Manual NumPy implementation

In [ ]:
def softmax_numpy(logits):                                       # logits: array of shape (batch, num_classes)
    shifted = logits - np.max(logits, axis=1, keepdims=True)     # Subtract row max for numerical stability
    exp_vals = np.exp(shifted)                                   # Exponentiate every shifted logit
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)    # Divide by row sum -> each row sums to 1


def cross_entropy_numpy(y_true_onehot, probs, eps=1e-12):        # y_true_onehot, probs: shape (batch, num_classes)
    probs = np.clip(probs, eps, 1.0)                             # Avoid log(0) by clipping tiny probabilities
    per_sample = -np.sum(y_true_onehot * np.log(probs), axis=1)  # -sum(y * log p) for each sample
    return per_sample                                            # Return loss for each sample (not averaged)


def cross_entropy_from_logits_numpy(y_true_onehot, logits):      # Stable version using log-sum-exp
    shifted = logits - np.max(logits, axis=1, keepdims=True)     # Shift logits by row max
    log_sum_exp = np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))  # log(sum(exp(z - max)))
    log_probs = shifted - log_sum_exp                            # log softmax = (z - max) - logsumexp
    return -np.sum(y_true_onehot * log_probs, axis=1)            # Per-sample cross-entropy

## Step 3 – Small toy example (easy to check by hand)

In [ ]:
toy_logits = np.array([[2.0, 1.0, 0.1],                          # Sample 1: class 0 has highest score
                       [0.5, 2.5, 0.3]])                         # Sample 2: class 1 has highest score
toy_labels = np.array([[1, 0, 0],                                # True class of sample 1 = 0 (one-hot)
                       [0, 0, 1]], dtype=np.float32)             # True class of sample 2 = 2 (wrong guess by model)

toy_probs = softmax_numpy(toy_logits)                            # Manual softmax
print("Softmax probabilities:\n", toy_probs)                     # Show probabilities
print("Row sums (should be 1):", toy_probs.sum(axis=1))          # Check each row sums to 1
print("Cross-entropy per sample:", cross_entropy_numpy(toy_labels, toy_probs))  # Low for correct, high for wrong
print("Check sample 1 by hand: -log(p0) =", -np.log(toy_probs[0, 0]))          # Should match first loss value

## Step 4 – Why the max-shift matters (overflow demo)

In [ ]:
big_logits = np.array([[1000.0, 1001.0, 1002.0]])                # Very large logits

naive = np.exp(big_logits) / np.sum(np.exp(big_logits))          # Naive softmax without shifting
print("Naive softmax :", naive)                                  # Gives nan because exp(1000) = inf
print("Stable softmax:", softmax_numpy(big_logits))              # Works fine: [0.090, 0.245, 0.665]

## Step 5 – Load MNIST and get real logits from a small model

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()  # Load MNIST

x_train = x_train.reshape(-1, 784).astype("float32") / 255.0     # Flatten and normalize to [0, 1]
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0       # Same for test data

y_train_onehot = keras.utils.to_categorical(y_train, 10)         # Integer labels -> one-hot vectors (60000, 10)
y_test_onehot = keras.utils.to_categorical(y_test, 10)           # Same for test labels

tf.random.set_seed(0)                                            # Reproducibility
logit_model = keras.Sequential([                                 # Model that outputs RAW LOGITS (no softmax)
    keras.Input(shape=(784,)),                                   # 784 input pixels
    layers.Dense(128, activation="relu"),                        # Hidden layer
    layers.Dense(10),                                            # Output layer with NO activation -> logits
])
logit_model.compile(                                             # Configure training
    optimizer="adam",                                            # Adam optimizer
    loss=keras.losses.CategoricalCrossentropy(from_logits=True), # Loss applies softmax internally
    metrics=["accuracy"],                                        # Track accuracy
)
logit_model.fit(x_train, y_train_onehot, epochs=2, batch_size=128, verbose=2)  # Short training so logits are meaningful

In [ ]:
x_batch = x_test[:8]                                             # Take first 8 test images
y_batch = y_test_onehot[:8]                                      # Their one-hot labels
logits = logit_model.predict(x_batch, verbose=0)                 # Raw logits, shape (8, 10)
print("Logits shape:", logits.shape)                             # Confirm shape
print("Logits of first image:\n", logits[0])                     # Show raw scores for first image

## Step 6 – Compare NumPy softmax with `tf.nn.softmax`

In [ ]:
probs_np = softmax_numpy(logits)                                 # Manual softmax
probs_tf = tf.nn.softmax(logits).numpy()                         # TensorFlow softmax

print("NumPy softmax (image 0):", probs_np[0])                   # Manual probabilities
print("TF    softmax (image 0):", probs_tf[0])                   # TF probabilities
print("Max abs difference     :", np.max(np.abs(probs_np - probs_tf)))  # Should be ~1e-7
print("All close?             :", np.allclose(probs_np, probs_tf, atol=1e-6))  # True if implementations agree

## Step 7 – Compare NumPy cross-entropy with `tf.nn.softmax_cross_entropy_with_logits`

In [ ]:
ce_np = cross_entropy_numpy(y_batch, probs_np)                   # Manual CE: softmax first, then -log
ce_np_stable = cross_entropy_from_logits_numpy(y_batch, logits)  # Manual CE directly from logits (stable)
ce_tf = tf.nn.softmax_cross_entropy_with_logits(                 # TensorFlow fused softmax + cross-entropy
    labels=y_batch,                                              # One-hot true labels
    logits=logits,                                               # Raw logits (NOT probabilities)
).numpy()                                                        # Convert tensor to NumPy

print("Per-sample CE (NumPy)          :", ce_np)                 # Manual result
print("Per-sample CE (NumPy stable)   :", ce_np_stable)          # Stable manual result
print("Per-sample CE (tf.nn ... logits):", ce_tf)                # TF result
print("All close?                     :", np.allclose(ce_np, ce_tf, atol=1e-5))  # Verify match

## Step 8 – Compare with the Keras loss API (batch mean)

In [ ]:
keras_ce_logits = keras.losses.CategoricalCrossentropy(from_logits=True)  # Keras loss that takes logits
keras_ce_probs = keras.losses.CategoricalCrossentropy()                   # Keras loss that takes probabilities

print("Mean CE NumPy                  :", ce_np.mean())                   # Average over batch
print("Mean CE tf.nn (mean)           :", ce_tf.mean())                   # Average of TF per-sample values
print("Keras CE from_logits=True      :", float(keras_ce_logits(y_batch, logits)))    # Keras from logits
print("Keras CE on softmax probs      :", float(keras_ce_probs(y_batch, probs_tf)))   # Keras from probabilities

## Step 9 – Visualize softmax output for a few test images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))                  # 2 rows x 4 columns
for i in range(4):                                               # First 4 images
    axes[0, i].imshow(x_batch[i].reshape(28, 28), cmap="gray")   # Show the digit image
    axes[0, i].set_title(f"True label: {np.argmax(y_batch[i])}") # True class as title
    axes[0, i].axis("off")                                       # Hide axes ticks
    axes[1, i].bar(range(10), probs_np[i])                       # Bar chart of 10 softmax probabilities
    axes[1, i].set_xticks(range(10))                             # Tick for each class 0-9
    axes[1, i].set_ylim(0, 1)                                    # Probabilities are in [0, 1]
    axes[1, i].set_title(f"Pred: {np.argmax(probs_np[i])}, CE={ce_np[i]:.3f}")  # Predicted class and its loss
plt.tight_layout()                                               # Tidy layout
plt.show()                                                       # Show figure

## Expected observations
- NumPy softmax and `tf.nn.softmax` match to ~1e-7.
- NumPy cross-entropy and `tf.nn.softmax_cross_entropy_with_logits` match to ~1e-6.
- Confident correct predictions → CE close to 0; wrong predictions → large CE.

## Viva questions
1. Why subtract max before exponentiating? → Prevents overflow; result is mathematically identical.
2. Why pass logits (not probabilities) to `softmax_cross_entropy_with_logits`? → It applies softmax internally in a numerically stable way (log-sum-exp).
3. Categorical vs sparse categorical cross-entropy? → Categorical expects one-hot labels; sparse expects integer labels. Same value.
4. What is the CE if the model predicts the true class with probability 1? → 0.
5. What is the CE for a uniform prediction over 10 classes? → $-\log(0.1) \approx 2.303$.